<a href="https://colab.research.google.com/github/ludtmit/baseball_pitch_prediction/blob/main/P3/baseball_predict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Notes area

I think we should start with a general model that tries to predict all pitches using all previous pitches.

If we are really feeling ourselves we can try to make profiles of batters and pitchers to feed into the network but first lets get the basics down.

In [ ]:
pip install pybaseball

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pybaseball]


In [ ]:
#all imports
from pybaseball import statcast
from pybaseball import cache
import pandas
import torch


In [ ]:
"""
statcast(start_dt="2019-06-24", end_dt="2019-06-25").columns
Index(['pitch_type', 'game_date', 'release_speed', 'release_pos_x',
       'release_pos_z', 'player_name', 'batter', 'pitcher', 'events',
       'description', 'spin_dir', 'spin_rate_deprecated',
       'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des',
       'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type',
       'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x',
       'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b',
       'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y',
       'tfs_deprecated', 'tfs_zulu_deprecated', 'fielder_2', 'umpire', 'sv_id',
       'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot',
       'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed',
       'release_spin_rate', 'release_extension', 'game_pk', 'pitcher.1',
       'fielder_2.1', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6',
       'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y',
       'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle',
       'woba_value', 'woba_denom', 'babip_value', 'iso_value',
       'launch_speed_angle', 'at_bat_number', 'pitch_number', 'pitch_name',
       'home_score', 'away_score', 'bat_score', 'fld_score', 'post_away_score',
       'post_home_score', 'post_bat_score', 'post_fld_score',
       'if_fielding_alignment', 'of_fielding_alignment', 'spin_axis',
       'delta_home_win_exp', 'delta_run_exp'],
      dtype='object')
"""

'\nstatcast(start_dt="2019-06-24", end_dt="2019-06-25").columns\nIndex([\'pitch_type\', \'game_date\', \'release_speed\', \'release_pos_x\',\n       \'release_pos_z\', \'player_name\', \'batter\', \'pitcher\', \'events\',\n       \'description\', \'spin_dir\', \'spin_rate_deprecated\',\n       \'break_angle_deprecated\', \'break_length_deprecated\', \'zone\', \'des\',\n       \'game_type\', \'stand\', \'p_throws\', \'home_team\', \'away_team\', \'type\',\n       \'hit_location\', \'bb_type\', \'balls\', \'strikes\', \'game_year\', \'pfx_x\',\n       \'pfx_z\', \'plate_x\', \'plate_z\', \'on_3b\', \'on_2b\', \'on_1b\',\n       \'outs_when_up\', \'inning\', \'inning_topbot\', \'hc_x\', \'hc_y\',\n       \'tfs_deprecated\', \'tfs_zulu_deprecated\', \'fielder_2\', \'umpire\', \'sv_id\',\n       \'vx0\', \'vy0\', \'vz0\', \'ax\', \'ay\', \'az\', \'sz_top\', \'sz_bot\',\n       \'hit_distance_sc\', \'launch_speed\', \'launch_angle\', \'effective_speed\',\n       \'release_spin_rate\', \'rele

In [ ]:
cache.enable()

In [ ]:
data = statcast(start_dt="2026-03-25", end_dt="2026-09-27").columns

This is a large query, it may take a moment to complete


  0%|          | 0/187 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/usr/local/lib/python3.13/dist-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = data_copy[column].apply(pd.to_datetime, errors='ignore', format=date_format)
/usr/local/lib/python3.13/dist-packages/pybaseball/datahelpers/postprocessing.py:59: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  data_copy[column] = 

In [ ]:
dev_index = int(data.size * .70) + 1
eval_index = int(data.size * .85) + 1
training_data = data[:dev_index-1]
development_data = data[dev_index:eval_index - 1]
evaluation_data = data[eval_index:]

In [ ]:
# A (Vijayanidhi): load 2026 regular season
from pybaseball import statcast, cache
import pandas as pd
import numpy as np

cache.enable()
raw = statcast(start_dt="2026-03-25", end_dt="2026-09-27")
raw = raw[raw["game_type"] == "R"].copy()
print(f"{len(raw):,} pitches loaded")

This is a large query, it may take a moment to complete


100%|██████████| 187/187 [00:09<00:00, 20.43it/s]
/usr/local/lib/python3.13/dist-packages/pybaseball/statcast.py:85: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_data = pd.concat(dataframe_list, axis=0).convert_dtypes(convert_string=False)


716,792 pitches loaded


In [ ]:
# B (Vijayanidhi): 4 pitch categories + count, same_hand, prev1-prev5
CAT = {**{p: "fastball" for p in ["FF", "SI", "FC", "FA"]},
       **{p: "slider"   for p in ["SL", "ST", "SV"]},
       **{p: "breaking" for p in ["CU", "KC", "CS"]},
       **{p: "offspeed" for p in ["CH", "FS", "FO", "SC"]}}
raw["pitch_cat"] = raw["pitch_type"].map(CAT)

raw = raw.sort_values(["game_date", "game_pk", "at_bat_number", "pitch_number"]).reset_index(drop=True)

raw["cat_for_hist"] = raw["pitch_cat"].astype(object).where(raw["pitch_cat"].notna(), "other")
g = raw.groupby(["game_pk", "at_bat_number"], sort=False)["cat_for_hist"]
for k in range(1, 6):
    raw[f"prev{k}"] = g.shift(k).fillna("none")
raw["same_hand"] = (raw["stand"] == raw["p_throws"]).astype(int)
raw["count"] = raw["balls"].astype(str) + "-" + raw["strikes"].astype(str)

df = raw[raw["pitch_cat"].notna()].copy()
print(f"Usable pitches: {len(df):,}")
df[["pitch_type", "pitch_cat", "count", "same_hand", "prev1", "prev2"]].head(10)

Usable pitches: 712,632


,pitch_type,pitch_cat,count,same_hand,prev1,prev2
0,SI,fastball,0-0,0,none,none
1,CH,offspeed,0-1,0,fastball,none
2,FF,fastball,0-2,0,offspeed,fastball
3,CH,offspeed,1-2,0,fastball,offspeed
4,CH,offspeed,1-2,0,offspeed,fastball
5,FF,fastball,1-2,0,offspeed,offspeed
6,SI,fastball,0-0,1,none,none
7,CH,offspeed,0-1,1,fastball,none
8,CH,offspeed,1-1,1,offspeed,fastball
9,SI,fastball,2-1,1,offspeed,offspeed


In [ ]:
# C (Vijayanidhi): chronological 70/15/15 split by game
games = (df.drop_duplicates("game_pk")[["game_pk", "game_date"]]
           .sort_values(["game_date", "game_pk"])["game_pk"].to_numpy())
n = len(games)
train = df[df["game_pk"].isin(set(games[: int(0.70 * n)]))]
dev   = df[df["game_pk"].isin(set(games[int(0.70 * n): int(0.85 * n)]))]
test  = df[df["game_pk"].isin(set(games[int(0.85 * n):]))]

print(f"Games: {n:,}")
for name, part in [("train", train), ("dev", dev), ("test", test)]:
    print(f"{name:5s}: {len(part):>8,} pitches, {part['game_pk'].nunique():>5,} games, "
          f"{part['game_date'].min()} to {part['game_date'].max()}")
print("\nClass share (train):")
print(train["pitch_cat"].value_counts(normalize=True).round(3))

Games: 2,429
train:  498,004 pitches, 1,700 games, 2026-03-25 00:00:00 to 2026-08-04 00:00:00
dev  :  107,233 pitches,   364 games, 2026-08-04 00:00:00 to 2026-08-31 00:00:00
test :  107,395 pitches,   365 games, 2026-08-31 00:00:00 to 2026-09-27 00:00:00

Class share (train):
pitch_cat
fastball    0.553
slider      0.222
offspeed    0.145
breaking    0.081
Name: proportion, dtype: float64


In [ ]:
# D (Vijayanidhi): baselines on dev
global_mode = train["pitch_cat"].mode()[0]
p_mode = train.groupby("pitcher")["pitch_cat"].agg(lambda s: s.mode()[0])
pred = dev["pitcher"].map(p_mode).fillna(global_mode)
acc_pitcher = (pred.to_numpy() == dev["pitch_cat"].to_numpy()).mean()

pc_mode = train.groupby(["pitcher", "count"])["pitch_cat"].agg(lambda s: s.mode()[0])
pred_pc = pd.Series(list(zip(dev["pitcher"], dev["count"])), index=dev.index).map(pc_mode).fillna(pred)
acc_pitcher_count = (pred_pc.to_numpy() == dev["pitch_cat"].to_numpy()).mean()

print(f"Always '{global_mode}': {(dev['pitch_cat'] == global_mode).mean():.3f}")
print(f"Pitcher's most common (BASELINE): {acc_pitcher:.3f}")
print(f"Pitcher x count most common: {acc_pitcher_count:.3f}")
print(f"Dev pitches from pitchers unseen in train: {(~dev['pitcher'].isin(p_mode.index)).mean():.1%}")

Always 'fastball': 0.560
Pitcher's most common (BASELINE): 0.566
Pitcher x count most common: 0.568
Dev pitches from pitchers unseen in train: 4.9%


In [ ]:
# E (Vijayanidhi): one-hot encode count, matchup, previous pitches
FEATS = ["count", "prev1", "prev2", "prev3", "prev4", "prev5", "stand", "p_throws"]
CLASSES = ["fastball", "slider", "offspeed", "breaking"]

X_all = pd.get_dummies(df[FEATS].astype(str), dtype="float32")
X_all["same_hand"] = df["same_hand"].astype("float32")
y_all = df["pitch_cat"].astype(object).map({c: i for i, c in enumerate(CLASSES)}).astype("int64")

Xtr, ytr = X_all.loc[train.index].to_numpy(), y_all.loc[train.index].to_numpy()
Xdv, ydv = X_all.loc[dev.index].to_numpy(), y_all.loc[dev.index].to_numpy()
print(Xtr.shape, Xdv.shape)

(498004, 47) (107233, 47)


In [ ]:
# F (Vijayanidhi): general model (no pitcher/batter identity), compare to 56.6% baseline
import torch, torch.nn as nn
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

Xtr_t, ytr_t = torch.tensor(Xtr, device=device), torch.tensor(ytr, device=device)
Xdv_t, ydv_t = torch.tensor(Xdv, device=device), torch.tensor(ydv, device=device)

model = nn.Sequential(
    nn.Linear(Xtr.shape[1], 64), nn.ReLU(), nn.Dropout(0.1),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 4)).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(5):
    model.train()
    perm = torch.randperm(len(Xtr_t), device=device)
    for i in range(0, len(perm), 2048):
        b = perm[i:i + 2048]
        opt.zero_grad()
        loss = loss_fn(model(Xtr_t[b]), ytr_t[b])
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        logits = model(Xdv_t)
        dev_loss = loss_fn(logits, ydv_t).item()
        dev_acc = (logits.argmax(1) == ydv_t).float().mean().item()
    print(f"epoch {epoch+1}: train loss {loss.item():.3f} | dev loss {dev_loss:.3f} | dev acc {dev_acc:.3f}")

epoch 1: train loss 1.026 | dev loss 1.048 | dev acc 0.564
epoch 2: train loss 1.093 | dev loss 1.045 | dev acc 0.565
epoch 3: train loss 1.066 | dev loss 1.044 | dev acc 0.566
epoch 4: train loss 1.072 | dev loss 1.044 | dev acc 0.566
epoch 5: train loss 1.086 | dev loss 1.043 | dev acc 0.566
